In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import pandas as pd
import numpy as np

!pip install -q lets-plot
from lets_plot import *
LetsPlot.setup_html()

In [17]:
d_r = 0.03
sample_size_new = 616
incidence = 71794
prevalence = 71794 # 640488
# market share (0.01 2024, 0.04 2025, 0.05 2025): https://doi.org/10.1080/13696998.2024.2411877
uptake_rate = 0.1 #0.04

trial_years = 3
delayed_trial_years = 3 # 10.1001/jamanetworkopen.2025.2026
total_years = 10

In [18]:
enbs_paths = "/content/drive/MyDrive/Colab Notebooks/01_Research/01_Local_Confirmatory_RCT/03_output/03_enbs"

# main results
df_a_val_unit_price = pd.read_csv(f"{enbs_paths}/01_combination_a_unit_price.csv")

# Create the repeated sequence of relative prices
relative_prices = np.tile([-30, -25, -20, -15, -10, -5, 0, 5, 10, 15, 20, 25, 30], 10)

# Ensure the length matches the DataFrame
relative_prices = relative_prices[:len(df_a_val_unit_price)]

# Assign the new relative prices to the DataFrame
df_a_val_unit_price['relative_price'] = relative_prices

df_a_val_unit_price

,a_value,price_drug,EV_ct_pp,EV_nt_pp,EV_update_pp,relative_price
0,0.1,10.393419,25214.604218,42665.211993,54746.234014,-30
1,0.1,11.135806,25214.604218,40055.308154,53075.516274,-25
2,0.1,11.878193,25214.604218,37445.413339,51458.761394,-20
3,0.1,12.620580,25214.604218,34835.527581,49901.119411,-15
4,0.1,13.362967,25214.604218,32225.650908,48402.312408,-10
...,...,...,...,...,...,...
125,1.0,16.332516,25214.604218,21741.479907,27692.222708,10
126,1.0,17.074903,25214.604218,19131.658272,26865.321983,15
127,1.0,17.817290,25214.604218,16521.839538,26261.814050,20
128,1.0,18.559677,25214.604218,13912.023707,25848.863172,25


In [19]:
def calculate_policy_comparison(df_input=None,
                                trial_years=trial_years,
                                prevalence=prevalence,
                                sample_size_new=sample_size_new,
                                incidence=incidence,
                                d_r=d_r,
                                uptake_rate=uptake_rate,
                                total_years=total_years,
                                delayed_trial_years=delayed_trial_years):
    """
    Calculate policy comparisons for different healthcare scenarios.

    Parameters:
    -----------
    df_input : pandas DataFrame, optional
        Input dataframe with drug price data. Default is None (should be provided when calling).
    trial_years : int, optional
        Number of years for the trial period. Default is 5.
    prevalence : int, optional
        Initial disease prevalence. Default is 1,000,000.
    sample_size_new : int, optional
        Sample size for new treatments. Default is 1,000.
    incidence : int, optional
        Annual incidence of new cases. Default is 50,000.
    d_r : float, optional
        Discount rate (as a decimal). Default is 0.03 (3%).
    uptake_rate : float, optional
        Rate of treatment uptake. Default is 0.5 (50%).
    total_years : int, optional
        Total time horizon for analysis in years. Default is 10.
    delayed_trial_years : int, optional
        Additional delay years for Policy 3. Default is 2.

    Returns:
    --------
    pandas DataFrame
        Combined policy comparison dataframe with EV calculations.
    """
    if df_input is None:
        raise ValueError("df_input must be provided")

    # Create a copy to avoid modifying the original
    df_a_val_unit_price = df_input.copy()

    # Policy 1
    P1_df = df_a_val_unit_price.copy()
    P1_df['part1'] = (prevalence - sample_size_new) * P1_df['EV_ct_pp']
    P1_df['part2'] = (sample_size_new/2) * P1_df['EV_ct_pp']
    P1_df['part3'] = (sample_size_new/2) * P1_df['EV_nt_pp']
    P1_df['part4'] = np.sum(incidence * (1 / (1+d_r) ** np.arange(1, trial_years, 1))) * P1_df['EV_ct_pp']
    P1_df['part5'] = (
        P1_df['EV_update_pp'] * np.sum(incidence * uptake_rate * (1 / (1 + d_r) ** np.arange(trial_years, total_years, 1)))
        + P1_df['EV_ct_pp'] * np.sum(incidence * (1 - uptake_rate) * (1 / (1 + d_r) ** np.arange(trial_years, total_years, 1)))
    )
    P1_df['EV_total'] = P1_df['part1'] + P1_df['part2'] + P1_df['part3'] + P1_df['part4'] + P1_df['part5']
    P1_df = P1_df[['a_value', 'price_drug', 'relative_price','EV_total']]

    # Policy 2
    P2_df = df_a_val_unit_price.copy()
    P2_df['part1'] = (
        (prevalence - sample_size_new) * P2_df['EV_nt_pp'] * uptake_rate
        + (prevalence - sample_size_new) * P2_df['EV_ct_pp'] * (1 - uptake_rate)
    )
    P2_df['part2'] = (sample_size_new/2) * P2_df['EV_ct_pp']
    P2_df['part3'] = (sample_size_new/2) * P2_df['EV_nt_pp']
    P2_df['part4'] = (
        np.sum(incidence * (1 / (1+d_r) ** np.arange(1, trial_years+delayed_trial_years, 1))) * P2_df['EV_nt_pp'] * uptake_rate
        + np.sum(incidence * (1 / (1+d_r) ** np.arange(1, trial_years+delayed_trial_years, 1))) * P2_df['EV_ct_pp'] * (1 - uptake_rate)
    )
    P2_df['part5'] = (
        P2_df['EV_update_pp'] * np.sum(incidence * uptake_rate * (1 / (1 + d_r) ** np.arange(trial_years+delayed_trial_years, total_years, 1)))
        + P2_df['EV_ct_pp'] * np.sum(incidence * (1 - uptake_rate) * (1 / (1 + d_r) ** np.arange(trial_years+delayed_trial_years, total_years, 1)))
    )
    P2_df['EV_total'] = P2_df['part1'] + P2_df['part2'] + P2_df['part3'] + P2_df['part4'] + P2_df['part5']
    P2_df = P2_df[['a_value', 'price_drug', 'relative_price','EV_total']]

    # Policy 3
    P3_df = df_a_val_unit_price.copy()
    P3_df['part1'] = (
        (prevalence + np.sum(incidence * (1 / (1 + d_r) ** np.arange(1, total_years, 1)))) * P3_df['EV_nt_pp'] * uptake_rate
        + (prevalence + np.sum(incidence * (1 / (1 + d_r) ** np.arange(1, total_years, 1)))) * P3_df['EV_ct_pp'] * (1 - uptake_rate)
    )
    P3_df['EV_total'] = P3_df['part1']
    P3_df = P3_df[['a_value', 'price_drug', 'relative_price','EV_total']]

    # Policy 4
    P4_df = df_a_val_unit_price.copy()
    P4_df['part1'] = (prevalence + np.sum(incidence * (1 / (1 + d_r) ** np.arange(1, total_years, 1)))) * P4_df['EV_ct_pp']
    P4_df['EV_total'] = P4_df['part1']
    P4_df = P4_df[['a_value', 'price_drug', 'relative_price','EV_total']]

    # Combine all policies
    dfs = [P1_df, P2_df, P3_df, P4_df]

    # Add policy indicator to each DataFrame
    for i, df in enumerate(dfs):
        df['policy_indicator'] = i + 1

    # Concatenate the DataFrames
    df_policy = pd.concat(dfs)
    df_policy['policy_indicator'] = df_policy['policy_indicator'].map({
        1: 'Policy 1', 2: 'Policy 2', 3: 'Policy 3', 4: 'Policy 4'
    })

    # Reset the index
    df_policy = df_policy.reset_index(drop=True)

    # Convert EV to billions
    df_policy['EV_total'] = df_policy['EV_total'] / 1000000000

    return df_policy

# base case: 3 year trial, delayed 3 years
df_policy_3yrs_trial = calculate_policy_comparison(df_input=df_a_val_unit_price)

In [20]:
scaler = 1.5

# Add font family configuration
times_new_roman_theme = theme(
    # title=element_text(size=12*scaler, family="Times New Roman"),
    axis_text_x=element_text(size=10*scaler, family="Times New Roman", angle=0),
    axis_text_y=element_text(size=10*scaler, family="Times New Roman"),
    legend_text=element_text(size=12*scaler, family="Times New Roman"),
    legend_position="bottom",  # Move legend to the bottom
    # legend_title=element_blank(),  # Remove color legend title
    legend_title=element_text(size=12*scaler, family="Times New Roman"),
    plot_title=element_text(size=16, family="Times New Roman", face = "bold"),
    plot_subtitle=element_text(size=14, family="Times New Roman"),
    axis_title_x=element_text(size=12*scaler, family="Times New Roman"),
    axis_title_y=element_text(size=12*scaler, family="Times New Roman"),
    text=element_text(size=8, family="Times New Roman")
)

# color mapping
color_set = {
    "Policy 1": "#EF9A80",
    "Policy 2": "#58BDCC",
    "Policy 3": "#06A18A",
    "Policy 4": "#E54B37"
}

color_set_reduced = {
    "Policy 1": "#EF9A80",
    "Policy 2": "#58BDCC"
}

# Figure 2: Expected value of four policies over the power prior

In [21]:
# Figure 2: Expected value of four policies over the power prior

# from df_policy_3yrs_trial only keep relative price to be -10
df_point_price_p2 = df_policy_3yrs_trial[df_policy_3yrs_trial['relative_price'] == -10]

figure2_plot = (
    ggplot(df_point_price_p2, aes(x="a_value", y="EV_total", color="policy_indicator", group="policy_indicator")) +
    geom_point(size = 3.2) +
    geom_line() +

    scale_x_continuous(breaks=df_a_val_unit_price["a_value"].unique(), format="{.1f}") +
    scale_y_continuous(format="${.1f}") +
    scale_color_manual(values=color_set) +

    labs(
        x = "Power Prior Parameter (α)",
        y = "Expected Net Benefits (Billion US$)",
        color = "Policy Scenario"
    ) +
    theme_bw() +
    times_new_roman_theme
)

figure2_plot

In [22]:
df_policy_3yrs_trial.to_csv(f"{enbs_paths}/02_3yr_trial.csv", index=False)
df_point_price_p2.to_csv(f"{enbs_paths}/03_figure2_values.csv", index=False)



# Figure 3: The optimal policy as a function of power prior and trial length


In [23]:
# Create datasets for trial years 1 to 6
policy_dfs = {}
for year in range(1, 7):
    policy_df = calculate_policy_comparison(df_input=df_a_val_unit_price, trial_years=year)
    # Subset for price_drug > 13 and < 14
    policy_df_filtered = policy_df[(policy_df['price_drug'] > 13) & (policy_df['price_drug'] < 14)]
    # Store in dictionary with appropriate name
    policy_dfs[f"df_policy_{year}yrs_trial"] = policy_df_filtered

# Create the final summary dataframe
df_rows = []

# For each trial year dataset
for trial_yr, df_name in enumerate(policy_dfs.keys(), 1):
    df = policy_dfs[df_name]

    # Group by a_value and find the policy with the maximum EV_total
    best_policies = df.loc[df.groupby('a_value')['EV_total'].idxmax()]

    # Create rows for the summary dataframe
    for _, row in best_policies.iterrows():
        df_rows.append({
            'a_value': row['a_value'],
            'trial_yr': trial_yr,
            'best_policy': row['policy_indicator'],
            'EV_total': row['EV_total']
        })

# Create the final dataframe
best_policies_p3 = pd.DataFrame(df_rows)

# Sort by a_value and trial_yr for better organization
best_policies_p3 = best_policies_p3.sort_values(by=['a_value', 'trial_yr']).reset_index(drop=True)

best_policies_p3['label'] = best_policies_p3.apply(
    lambda row: f"{row['best_policy']}", axis=1
)

# change policy 1 to be 1, policy 2 to be 2, etc.
label_map = {
    'Policy 1': 1,
    'Policy 2': 2,
    'Policy 3': 3,
    'Policy 4': 4
}

best_policies_p3['label'] = (
    best_policies_p3['label']
      .map(label_map)
)

In [24]:
figure3_plot = (
    ggplot(best_policies_p3, aes(x='a_value',y='trial_yr')) +
    geom_tile(aes(fill='best_policy')) +
    # geom_text(aes(label='label')) +
    scale_fill_manual(values=color_set_reduced) +
    scale_x_continuous(breaks=best_policies_p3["a_value"].unique(), format="{.1f}") +
    labs(
        x = "Power Prior Parameter (α)",
        y = "Standard Trial Duration (Years)",
        fill = "Policy Scenario"
    ) +
    theme_minimal() +
    times_new_roman_theme
)

figure3_plot

# Figure 4: The optimal policy as a function of power prior and delayed trial length


In [25]:
# Create datasets for trial years 1 to 6 (same as before)
policy_dfs = {}
for year in range(1, 7):
    policy_df = calculate_policy_comparison(df_input=df_a_val_unit_price, delayed_trial_years=year)
    policy_df_filtered = policy_df[(policy_df['price_drug'] > 13) & (policy_df['price_drug'] < 14)]
    policy_dfs[f"df_policy_{year}yrs_trial"] = policy_df_filtered

# Create the final summary dataframe
df_rows = []

# For each trial year dataset
for delayed_yr, df_name in enumerate(policy_dfs.keys(), 1):
    df = policy_dfs[df_name]

    # Group by a_value and find the policy with the maximum EV_total
    best_policies = df.loc[df.groupby('a_value')['EV_total'].idxmax()]

    # Create rows for the summary dataframe
    for _, row in best_policies.iterrows():
        df_rows.append({
            'a_value': row['a_value'],
            'delayed_yr': delayed_yr,
            'best_policy': row['policy_indicator'],
            'EV_total': row['EV_total']
        })

# Create the final dataframe
best_policies_p4 = pd.DataFrame(df_rows)

# Sort by a_value and trial_yr for better organization
best_policies_p4 = best_policies_p4.sort_values(by=['a_value', 'delayed_yr']).reset_index(drop=True)

best_policies_p4['label'] = best_policies_p4.apply(
    lambda row: f"{row['best_policy']}", axis=1
)

best_policies_p4['label'] = (
    best_policies_p4['label']
      .map(label_map)
)

# Create the heatmap plot
figure4_plot = (
    ggplot(best_policies_p4, aes(x='a_value', y='delayed_yr')) +
    geom_tile(aes(fill='best_policy')) +
    # geom_text(aes(label='label')) +
    scale_fill_manual(values=color_set_reduced) +
    scale_x_continuous(breaks=best_policies_p4["a_value"].unique(), format="{.1f}") +
    labs(
        x = "Power Prior Parameter (α)",
        y = "Additional Trial Delay (Years)",
        fill = "Policy Scenario"
    ) +
    theme_minimal() +
    times_new_roman_theme
)

figure4_plot

# Figure 5: The optimal policy as a function of power prior and price

In [26]:
# First, identify the best policy for each a_value and relative_price combination
best_policies_p5 = df_policy_3yrs_trial.loc[
    df_policy_3yrs_trial.groupby(['a_value', 'relative_price'])['EV_total'].idxmax()
]

# Format the label with dollar sign and "B" suffix
best_policies_p5['label'] = best_policies_p5.apply(
    lambda row: f"{row['policy_indicator']}", axis=1
)

best_policies_p5['label'] = (
    best_policies_p5['label']
      .map(label_map)
)

In [27]:
figure5_plot = (
    ggplot(best_policies_p5, aes(x='a_value', y='relative_price')) +
    geom_tile(aes(fill='policy_indicator')) +
    # geom_text(aes(label='label')) +
    scale_fill_manual(values=color_set_reduced) +
    scale_x_continuous(breaks=best_policies_p5["a_value"].unique(), format="{.1f}") +
    scale_y_continuous(breaks=best_policies_p5["relative_price"].unique(), format="{.0f}") +
    labs(
        x = "Power Prior Parameter (α)",
        y = "Price Deviation from Value-Based Price (%)",
        fill = "Policy Scenario"
    ) +
    theme_minimal() +
    times_new_roman_theme
)

figure5_plot

In [52]:
all_plots = gggrid([
    figure3_plot + theme(legend_position="none"),
    figure4_plot + theme(legend_position="none"),
    figure5_plot + theme(legend_position="none")
    ])

all_plots

In [54]:
!pip install -q CairoSVG
ggsave(plot=figure2_plot, filename=f"{enbs_paths}/02_over_a_value.pdf", dpi=1500, w=4*4, h=3*4, unit='in')
ggsave(plot=figure3_plot, filename=f"{enbs_paths}/03_best_a_trial_length.pdf", dpi=1500, w=4*4, h=3*4, unit='in')
ggsave(plot=figure4_plot, filename=f"{enbs_paths}/04_best_a_delayed_trial.pdf", dpi=1500, w=4*4, h=3*4, unit='in')
ggsave(plot=figure5_plot, filename=f"{enbs_paths}/05_best_a_price_relative.pdf", dpi=1500, w=4*4, h=3*4, unit='in')

ggsave(plot=all_plots, filename=f"{enbs_paths}/06_all_plots.pdf", dpi=1500, w=4*6, h=3*6, unit='in')

CairoError: cairo returned CAIRO_STATUS_INVALID_SIZE: b'invalid value (typically too big) for the size of the input (surface, pattern, etc.)'

In [55]:
ggsave(plot=figure3_plot, filename=f"{enbs_paths}/03_best_a_trial_length.png", dpi=1500, w=4*4, h=3*4, unit='in')
ggsave(plot=all_plots, filename=f"{enbs_paths}/06_all_plots.png", dpi=1500, w=4*4, h=3*4, unit='in')

'/content/drive/MyDrive/Colab Notebooks/01_Research/01_Local_Confirmatory_RCT/03_output/03_enbs/06_all_plots.png'